In [9]:
!pip install pandas openpyxl matplotlib seaborn

zsh:1: command not found: pip


In [10]:
import os 

print("Diretório atual:", os.getcwd())

try:
    print ("Arquivos em data:", os.listdir('../data'))
except Exception as e: 
    print("Erro ao tentar acessar '../data':", e)

    # tentar olhar a pasta 'data' direto do ponto atual
    try:
        print("Tentativa alternativa (sem ../):", os.listdir('data'))
    except:
        pass



Diretório atual: /Users/thiagomota/Documents/MdiasCase/notebooks
Arquivos em data: ['dataset_manutencao_estagio.csv', 'base_de_dados_-_perdas_nov24.xlsx']


In [11]:
import pandas as pd 

# penso que o VScode precisa sair da pasta "notebooks" e enxergar a pasta 'data' 
df_perdas = pd.read_excel('../data/base_de_dados_-_perdas_nov24.xlsx')

# os dados carregaram de forma correta? Checagem
df_perdas.head()

,ORDEM DE PRODUÇÃO,DATA DA PERDA,LINHA / PROCESSO,SEÇÃO / ENVASE,CÓDIGO DO PRODUTO,DESCRIÇÃO PRODUTO,HORÁRIO,TURNO DE PRODUÇÃO,TIPO DE PERDA,TIPO DE EMBALAGEM (Somente para indicador perda de embalagem),DEFEITO QUE GEROU A PERDA,CÓD. DEFEITO,QUANTIDADE (KG)
0,1046893,2024-11-01,LB04,-,30258,"RICH AMORI RECH CHOC 40X71,2G",00:01:00,3,Sobrepeso,Não se aplica,NaN,-,0.430
1,1046893,2024-11-01,LB04,Embalagem,30258,"RICH AMORI RECH CHOC 40X71,2G",00:01:00,3,Reprocesso,Não se aplica,Defeito 2,DP.PBI.041,1.000
2,1047749,2024-11-03,LB04,Empilhamento,30260,"RICH ESCUR RECH 40X71,2G",06:00:00,1,Reprocesso,Não se aplica,Defeito 2,DP.PBI.041,94.500
3,1047749,2024-11-03,LB04,Embalagem,30260,"RICH ESCUR RECH 40X71,2G",06:00:00,1,Perda de Embalagem,PRIMÁRIA,Defeito 10,DE.PBI.001,0.222
4,1047749,2024-11-03,LB04,Embalagem,30260,"RICH ESCUR RECH 40X71,2G",06:00:00,1,Perda de Embalagem,PRIMÁRIA,Defeito 10,DE.PBI.001,0.267


In [18]:
# Qual o total de perdas (em kg) registrado em novembro? Qual o tipo de perda com maior volume?

# somar coluna de quantidades
total_kg = df_perdas['QUANTIDADE (KG)'].sum()
print(f"O total de perdas em novembro foi: {total_kg:,.2f} KG\n")

# encontrar perda maior
ranking_tipos = df_perdas.groupby('TIPO DE PERDA')['QUANTIDADE (KG)'].sum().sort_values(ascending = False)
print("Ranking por Tipo de Perda (em KG):")
print(ranking_tipos)

O total de perdas em novembro foi: 19,736.90 KG

Ranking por Tipo de Perda (em KG):
TIPO DE PERDA
Reprocesso            10898.500
Sobrepeso              6227.879
Descarte               2317.390
Perda de Embalagem      293.127
Name: QUANTIDADE (KG), dtype: float64


In [23]:
# preciso contar quantos dias diferentes exsitem na tabela
 
dias_com_registro = df_perdas['DATA DA PERDA'].nunique()
print(f"Quantidade de dias com registro no mês: {dias_com_registro} dias\n")

# preciso calcular a média 
perdas_diarias = df_perdas.groupby('DATA DA PERDA')['QUANTIDADE (KG)'].sum()
media_diaria = perdas_diarias.mean()

print(f"Média de perda por dia: {media_diaria:,.2f} KG\n")

print("Os 5 dias com maiores volumes de perda (Acima da Média):")
print(perdas_diarias.sort_values(ascending=False).head(5))


Quantidade de dias com registro no mês: 24 dias

Média de perda por dia: 822.37 KG

Os 5 dias com maiores volumes de perda (Acima da Média):
DATA DA PERDA
2024-11-19    1495.673
2024-11-24    1368.729
2024-11-14    1083.935
2024-11-06    1032.922
2024-11-25    1002.094
Name: QUANTIDADE (KG), dtype: float64


In [28]:
# Entender os produtos diferentes
# Ver a quantidade de codigos diferentes de produto -> nunigue()

total_produtos = df_perdas['CÓDIGO DO PRODUTO'].nunique()
print(f"Quantidade de produtos diferentes produzidos: {total_produtos}\n")

# Produto que concentra maior parte das perdas
# .idxmax() descobre o nome do produto com a maior perda
perdas_por_produto = df_perdas.groupby('DESCRIÇÃO PRODUTO')['QUANTIDADE (KG)'].sum()
produto_vilao = perdas_por_produto.idxmax()
quilos_vilao = perdas_por_produto.max()

print(f"O produto com a maior parte das perdas: {produto_vilao}")
print(f"Volume perdido: {quilos_vilao:,.2f} KG")


Quantidade de produtos diferentes produzidos: 11

O produto com a maior parte das perdas: RICH AMORI RECH CHOC 40X35,6G
Volume perdido: 5,049.06 KG


In [34]:
# como posso encontrar problemas de qualidade dos dados?

print("--- 1. QUANTIDADE DE CAMPOS EM BRANCO (VALORES NULOS) ---")
print(df_perdas.isnull().sum())
print("\n" + "="*50 + "\n")

print("--- 2. VERIFICAÇÃO DE VALORES VALORES SUSPEITOS EM QUANTIDADE ---")
print(df_perdas['QUANTIDADE (KG)'].describe())
print("\n" + "="*50 + "\n")

print("--- 3. REGISTROS INCONSISTENTES NA COLUNA SEÇÃO/ENVASE ---")
print(df_perdas['SEÇÃO / ENVASE'].value_counts())

--- 1. QUANTIDADE DE CAMPOS EM BRANCO (VALORES NULOS) ---
ORDEM DE PRODUÇÃO                                                 0
DATA DA PERDA                                                     0
LINHA / PROCESSO                                                  0
SEÇÃO / ENVASE                                                    0
CÓDIGO DO PRODUTO                                                 0
DESCRIÇÃO PRODUTO                                                 0
HORÁRIO                                                           0
TURNO DE PRODUÇÃO                                                 0
TIPO DE PERDA                                                     0
TIPO DE EMBALAGEM (Somente para indicador perda de embalagem)     0
DEFEITO QUE GEROU A PERDA                                        85
CÓD. DEFEITO                                                      0
QUANTIDADE (KG)                                                   0
dtype: int64


--- 2. VERIFICAÇÃO DE VALORES VALORES SUSPE

In [36]:
df_perdas.to_csv('../data/dataset_perdas_looker.csv', index=False, encoding='utf-8-sig')
print("Sucesso! Arquivo gerado dentro da pasta data do seu projeto.")

Sucesso! Arquivo gerado dentro da pasta data do seu projeto.
